# aFRR VWAP Validation (Regelleistung 15-min)

This notebook validates hourly aFRR VWAP against 15-minute price/volume rows for two windows:

1. **Pre-PICASSO:** 2021-03-01 10:00 to 16:00 UTC  
2. **Post-PICASSO:** 2022-08-01 10:00 to 16:00 UTC

Hourly VWAP is repeated on each quarter-hour row (same hour) for visual comparison.

In [19]:
from pathlib import Path
import polars as pl
import pandas as pd

# Robustly locate the project data file by walking up parent directories.
REL = Path("data/raw/regelleistung_15min/afrr_price_volume_15min.parquet")
search_roots = [Path.cwd(), *Path.cwd().parents]
PATH_15M = None
for root in search_roots:
    candidate = (root / REL).resolve()
    if candidate.exists():
        PATH_15M = candidate
        break

if PATH_15M is None:
    tried = "\n".join(str((r / REL).resolve()) for r in search_roots)
    raise FileNotFoundError(
        "Missing 15-min parquet. Tried:\n" + tried
    )

df = pl.read_parquet(PATH_15M).with_columns(
    pl.col("timestamp_utc").cast(pl.Datetime(time_unit="us", time_zone="UTC"), strict=False)
).sort("timestamp_utc")

In [20]:
def first_existing(columns, candidates):
    for c in candidates:
        if c in columns:
            return c
    return None


price_pos_col = first_existing(
    df.columns,
    [
        "afrr_avg_activation_price_pos",
        "afrr_activation_avg_price_pos",
        "afrr_marginal_activation_price_pos",
        "afrr_activation_marginal_price_pos",
        "arbeitspreis_pos",
    ],
)
price_neg_col = first_existing(
    df.columns,
    [
        "afrr_avg_activation_price_neg",
        "afrr_activation_avg_price_neg",
        "afrr_marginal_activation_price_neg",
        "afrr_activation_marginal_price_neg",
        "arbeitspreis_neg",
    ],
)
vol_pos_col = first_existing(df.columns, ["afrr_activated_mw_pos", "activated_volume_pos_mw", "abgerufene_arbeit_pos"])
vol_neg_col = first_existing(df.columns, ["afrr_activated_mw_neg", "activated_volume_neg_mw", "abgerufene_arbeit_neg"])
official_avg_pos_col = first_existing(df.columns, ["afrr_avg_activation_price_pos", "afrr_activation_avg_price_pos", "durchschnittlicher_arbeitspreis_pos"])
official_avg_neg_col = first_existing(df.columns, ["afrr_avg_activation_price_neg", "afrr_activation_avg_price_neg", "durchschnittlicher_arbeitspreis_neg"])

print("price_pos_col:", price_pos_col)
print("price_neg_col:", price_neg_col)
print("vol_pos_col:", vol_pos_col)
print("vol_neg_col:", vol_neg_col)
print("official_avg_pos_col:", official_avg_pos_col)
print("official_avg_neg_col:", official_avg_neg_col)


def build_window_table(df_in: pl.DataFrame, start_utc: str, end_utc: str) -> pd.DataFrame:
    cutoff = pd.Timestamp("2022-06-22T22:00:00Z")
    start_ts = pd.Timestamp(start_utc)
    end_ts = pd.Timestamp(end_utc)

    w = df_in.filter(
        (pl.col("timestamp_utc") >= pl.lit(start_ts))
        & (pl.col("timestamp_utc") < pl.lit(end_ts))
    ).sort("timestamp_utc")

    needed = ["timestamp_utc"]
    for c in [vol_pos_col, vol_neg_col, price_pos_col, price_neg_col, official_avg_pos_col, official_avg_neg_col]:
        if c is not None and c not in needed:
            needed.append(c)
    w = w.select(needed)

    base_exprs = [pl.col("timestamp_utc").dt.truncate("1h").alias("hour_utc")]

    if price_pos_col:
        base_exprs.append(
            pl.when(pl.col("timestamp_utc") < pl.lit(cutoff))
            .then(pl.col(price_pos_col).forward_fill().over(pl.col("timestamp_utc").dt.truncate("1h")))
            .otherwise(pl.col(price_pos_col))
            .alias("price_pos_ff")
        )
    if price_neg_col:
        base_exprs.append(
            pl.when(pl.col("timestamp_utc") < pl.lit(cutoff))
            .then(pl.col(price_neg_col).forward_fill().over(pl.col("timestamp_utc").dt.truncate("1h")))
            .otherwise(pl.col(price_neg_col))
            .alias("price_neg_ff")
        )

    w2 = w.with_columns(base_exprs)

    wc_exprs = []
    if price_pos_col and vol_pos_col and "price_pos_ff" in w2.columns:
        wc_exprs.append((pl.col("price_pos_ff") * pl.col(vol_pos_col).cast(pl.Float64, strict=False)).alias("wc_pos"))
    if price_neg_col and vol_neg_col and "price_neg_ff" in w2.columns:
        wc_exprs.append((pl.col("price_neg_ff") * pl.col(vol_neg_col).cast(pl.Float64, strict=False)).alias("wc_neg"))
    if wc_exprs:
        w2 = w2.with_columns(wc_exprs)

    agg_exprs = []
    if vol_pos_col:
        agg_exprs.append(pl.col(vol_pos_col).cast(pl.Float64, strict=False).sum().alias("sum_vol_pos"))
    if vol_neg_col:
        agg_exprs.append(pl.col(vol_neg_col).cast(pl.Float64, strict=False).sum().alias("sum_vol_neg"))
    if "wc_pos" in w2.columns:
        agg_exprs.append(pl.col("wc_pos").sum().alias("sum_wc_pos"))
    if "wc_neg" in w2.columns:
        agg_exprs.append(pl.col("wc_neg").sum().alias("sum_wc_neg"))
    if official_avg_pos_col:
        agg_exprs.append(pl.col(official_avg_pos_col).cast(pl.Float64, strict=False).mean().alias("official_avg_pos_hourly"))
    if official_avg_neg_col:
        agg_exprs.append(pl.col(official_avg_neg_col).cast(pl.Float64, strict=False).mean().alias("official_avg_neg_hourly"))

    hourly = (
        w2.group_by_dynamic(
            index_column="timestamp_utc",
            every="1h",
            period="1h",
            closed="left",
            label="left",
        )
        .agg(agg_exprs)
        .sort("timestamp_utc")
        .rename({"timestamp_utc": "hour_utc"})
    )

    add_hourly = []
    if "sum_wc_pos" in hourly.columns and "sum_vol_pos" in hourly.columns:
        add_hourly.append(
            pl.when(pl.col("sum_vol_pos").is_null() | (pl.col("sum_vol_pos") == 0))
            .then(pl.lit(float("nan")))
            .otherwise(pl.col("sum_wc_pos") / pl.col("sum_vol_pos"))
            .alias("afrr_vwap_pos_eur_mwh")
        )
    if "sum_wc_neg" in hourly.columns and "sum_vol_neg" in hourly.columns:
        add_hourly.append(
            pl.when(pl.col("sum_vol_neg").is_null() | (pl.col("sum_vol_neg") == 0))
            .then(pl.lit(float("nan")))
            .otherwise(pl.col("sum_wc_neg") / pl.col("sum_vol_neg"))
            .alias("afrr_vwap_neg_eur_mwh")
        )
    if add_hourly:
        hourly = hourly.with_columns(add_hourly)

    join_cols = [c for c in ["hour_utc", "afrr_vwap_pos_eur_mwh", "afrr_vwap_neg_eur_mwh", "official_avg_pos_hourly", "official_avg_neg_hourly"] if c in hourly.columns]
    joined = w2.join(hourly.select(join_cols), on="hour_utc", how="left").sort("timestamp_utc")

    rename = {}
    if vol_pos_col:
        rename[vol_pos_col] = "vol_15m_pos"
    if vol_neg_col:
        rename[vol_neg_col] = "vol_15m_neg"
    if price_pos_col:
        rename[price_pos_col] = "price_15m_pos_raw"
    if price_neg_col:
        rename[price_neg_col] = "price_15m_neg_raw"
    if "price_pos_ff" in joined.columns:
        rename["price_pos_ff"] = "price_15m_pos_ffill"
    if "price_neg_ff" in joined.columns:
        rename["price_neg_ff"] = "price_15m_neg_ffill"

    out = joined.rename(rename)
    out_cols = [
        "timestamp_utc",
        "vol_15m_pos",
        "vol_15m_neg",
        "price_15m_pos_raw",
        "price_15m_neg_raw",
        "price_15m_pos_ffill",
        "price_15m_neg_ffill",
        "official_avg_pos_hourly",
        "official_avg_neg_hourly",
        "afrr_vwap_pos_eur_mwh",
        "afrr_vwap_neg_eur_mwh",
    ]
    out_cols = [c for c in out_cols if c in out.columns]
    return out.select(out_cols).to_pandas().set_index("timestamp_utc")


price_pos_col: afrr_avg_activation_price_pos
price_neg_col: afrr_avg_activation_price_neg
vol_pos_col: afrr_activated_mw_pos
vol_neg_col: afrr_activated_mw_neg
official_avg_pos_col: afrr_avg_activation_price_pos
official_avg_neg_col: afrr_avg_activation_price_neg


In [21]:
pre_table = build_window_table(df, "2021-03-01T10:00:00Z", "2021-03-01T16:00:00Z")
post_table = build_window_table(df, "2022-08-01T10:00:00Z", "2022-08-01T16:00:00Z")

print("Pre-PICASSO window (2021-03-01 10:00-16:00 UTC)")
display(pre_table)

print("Post-PICASSO window (2022-08-01 10:00-16:00 UTC)")
display(post_table)

Pre-PICASSO window (2021-03-01 10:00-16:00 UTC)


,vol_15m_pos,vol_15m_neg,price_15m_pos_raw,price_15m_neg_raw,price_15m_pos_ffill,price_15m_neg_ffill,official_avg_pos_hourly,official_avg_neg_hourly,afrr_vwap_pos_eur_mwh,afrr_vwap_neg_eur_mwh
timestamp_utc,,,,,,,,,,
2021-03-01 10:00:00+00:00,46.272,11.424,2554.23,-124.73,2554.23,-124.73,2554.23,-124.73,2554.23,-124.73
2021-03-01 10:15:00+00:00,0.780,100.752,NaN,NaN,2554.23,-124.73,2554.23,-124.73,2554.23,-124.73
2021-03-01 10:30:00+00:00,6.404,33.472,NaN,NaN,2554.23,-124.73,2554.23,-124.73,2554.23,-124.73
2021-03-01 10:45:00+00:00,2.044,325.220,NaN,NaN,2554.23,-124.73,2554.23,-124.73,2554.23,-124.73
2021-03-01 11:00:00+00:00,74.340,95.712,460.21,-192.70,460.21,-192.70,460.21,-192.70,460.21,-192.70
2021-03-01 11:15:00+00:00,9.624,103.692,NaN,NaN,460.21,-192.70,460.21,-192.70,460.21,-192.70
2021-03-01 11:30:00+00:00,16.248,197.724,NaN,NaN,460.21,-192.70,460.21,-192.70,460.21,-192.70
2021-03-01 11:45:00+00:00,1.324,462.744,NaN,NaN,460.21,-192.70,460.21,-192.70,460.21,-192.70
2021-03-01 12:00:00+00:00,263.752,48.708,460.21,-192.70,460.21,-192.70,460.21,-192.70,460.21,-192.70


Post-PICASSO window (2022-08-01 10:00-16:00 UTC)


,vol_15m_pos,vol_15m_neg,price_15m_pos_raw,price_15m_neg_raw,price_15m_pos_ffill,price_15m_neg_ffill,official_avg_pos_hourly,official_avg_neg_hourly,afrr_vwap_pos_eur_mwh,afrr_vwap_neg_eur_mwh
timestamp_utc,,,,,,,,,,
2022-08-01 10:00:00+00:00,352.536,0.000,1273.71,-141.51,1273.71,-141.51,975.5725,-115.9125,1072.074616,-105.630114
2022-08-01 10:15:00+00:00,160.332,0.004,892.14,-116.08,892.14,-116.08,975.5725,-115.9125,1072.074616,-105.630114
2022-08-01 10:30:00+00:00,6.984,0.348,879.36,-105.51,879.36,-105.51,975.5725,-115.9125,1072.074616,-105.630114
2022-08-01 10:45:00+00:00,190.184,0.000,857.08,-100.55,857.08,-100.55,975.5725,-115.9125,1072.074616,-105.630114
2022-08-01 11:00:00+00:00,231.264,0.008,844.96,-102.67,844.96,-102.67,899.7025,-100.4075,884.144017,-101.434029
2022-08-01 11:15:00+00:00,151.532,5.844,915.50,-97.80,915.50,-97.80,899.7025,-100.4075,884.144017,-101.434029
2022-08-01 11:30:00+00:00,62.488,3.300,952.88,-98.56,952.88,-98.56,899.7025,-100.4075,884.144017,-101.434029
2022-08-01 11:45:00+00:00,11.496,26.340,885.47,-102.60,885.47,-102.60,899.7025,-100.4075,884.144017,-101.434029
2022-08-01 12:00:00+00:00,0.688,2.820,3111.15,-806.28,3111.15,-806.28,1560.8025,-355.6350,1393.581679,-200.400868
